# Task 2 - Recommendation System Development

Load and integrate the allocated customer, product and transaction data. Build and evaluate a popularity-based recommender plus one additional method. Keep business interpretation for Task 3.

In [5]:
from pathlib import Path
import numpy as np
import pandas as pd
from sklearn.metrics.pairwise import cosine_similarity

In [6]:
DATA_DIR = Path('../data')
customers = pd.read_csv(DATA_DIR / 'Customers_repeat.csv')
products = pd.read_csv(DATA_DIR / 'Products_sort.csv')
transactions = pd.read_csv(DATA_DIR / 'Transactions_made.csv')
customers.head(), products.head(), transactions.head()

(   CustomerID AgeGroup Region LoyaltyLevel  MembershipYears
 0           1    18-25  North       Bronze                1
 1           2    18-25   East       Bronze                2
 2           3      50+   East       Bronze                5
 3           4    36-50   East       Silver               12
 4           5    26-35   East       Silver               13,
    ProductID     Category       Brand   Price  Rating  PopularityScore
 0          1      Kitchen     FitLife  460.84    4.46                2
 1          2  Electronics     TechMax  211.35    3.39                1
 2          3         Home     HomePro  337.91    3.31                2
 3          4       Gaming     TechMax  323.63    3.66                1
 4          5      kitchen  PrimeGoods  202.09    2.52                4,
    CustomerID  ProductID  PurchaseCount  LastPurchaseDays    Spend
 0           1        120              2                92    36.24
 1           2         55              4               293  1872

In [7]:
# Validate keys and missing values before joining tables.
for name, frame in {'customers': customers, 'products': products, 'transactions': transactions}.items():
    print(name, frame.shape)
    display(frame.isna().sum().sort_values(ascending=False).head())
print('customer IDs missing from customer table:', (~transactions['CustomerID'].isin(customers['CustomerID'])).sum())
print('product IDs missing from product table:', (~transactions['ProductID'].isin(products['ProductID'])).sum())
interaction_matrix = transactions.pivot_table(index='CustomerID', columns='ProductID', values='PurchaseCount', aggfunc='sum', fill_value=0)
interaction_matrix.shape

customers (1000, 5)


CustomerID         0
AgeGroup           0
Region             0
LoyaltyLevel       0
MembershipYears    0
dtype: int64

products (200, 6)


Rating       15
ProductID     0
Category      0
Brand         0
Price         0
dtype: int64

transactions (18711, 5)


CustomerID          0
ProductID           0
PurchaseCount       0
LastPurchaseDays    0
Spend               0
dtype: int64

customer IDs missing from customer table: 0
product IDs missing from product table: 0


(1000, 200)

## First three Task 2 requirements

The tables are joined using their customer and product IDs. Customer value is then summarised using total spend and total purchase frequency.

In [8]:
# Integrate the three tables.
integrated_data = transactions.merge(customers, on='CustomerID', how='left')
integrated_data = integrated_data.merge(products, on='ProductID', how='left')

print('Integrated rows and columns:', integrated_data.shape)
print('Missing customer details:', integrated_data['AgeGroup'].isna().sum())
print('Missing product details:', integrated_data['Category'].isna().sum())

# Top 10 customers by spend and purchase frequency.
customer_summary = transactions.groupby('CustomerID').agg(
    TotalSpend=('Spend', 'sum'),
    TotalPurchases=('PurchaseCount', 'sum')
).sort_values('TotalSpend', ascending=False)

display(customer_summary.head(10))
display(customer_summary.sort_values('TotalPurchases', ascending=False).head(10))

Integrated rows and columns: (18711, 14)
Missing customer details: 0
Missing product details: 0


,TotalSpend,TotalPurchases
CustomerID,,
971,32222.52,83
174,29802.65,78
227,28402.94,81
184,28213.86,80
761,27805.10,77
611,27618.73,77
840,27082.53,81
434,26933.14,68
880,26651.43,71


,TotalSpend,TotalPurchases
CustomerID,,
79,26046.02,89
95,19797.80,87
553,9573.32,87
373,25882.32,85
831,17269.04,84
265,8317.85,83
971,32222.52,83
340,9609.71,82
448,13576.84,82


## Product performance and popularity recommender

Product, category and brand performance are compared using transaction spend. The popularity recommender suggests highly purchased products that the customer has not already bought.

In [9]:
# Three groupby analyses: product, category and brand performance.
product_analysis = integrated_data.groupby(['ProductID', 'Category', 'Brand']).agg(
    TotalRevenue=('Spend', 'sum'),
    TotalPurchases=('PurchaseCount', 'sum')
).sort_values('TotalRevenue', ascending=False)

category_analysis = integrated_data.groupby('Category').agg(
    TotalRevenue=('Spend', 'sum'),
    TotalPurchases=('PurchaseCount', 'sum')
).sort_values('TotalRevenue', ascending=False)

brand_analysis = integrated_data.groupby('Brand').agg(
    TotalRevenue=('Spend', 'sum'),
    TotalPurchases=('PurchaseCount', 'sum')
).sort_values('TotalRevenue', ascending=False)

display(product_analysis.head(10))
display(category_analysis)
display(brand_analysis)

# Two pivot tables.
region_category_pivot = pd.pivot_table(
    integrated_data,
    values='Spend',
    index='Region',
    columns='Category',
    aggfunc='sum',
    fill_value=0
)
loyalty_category_pivot = pd.pivot_table(
    integrated_data,
    values='Spend',
    index='LoyaltyLevel',
    columns='Category',
    aggfunc='sum',
    fill_value=0
)

display(region_category_pivot)
display(loyalty_category_pivot)

# Popularity-based recommender.
popular_products = product_analysis.reset_index().sort_values(
    ['TotalPurchases', 'TotalRevenue'], ascending=False
)

def popularity_recommendations(customer_id, number_of_recommendations=5):
    purchased_products = set(
        transactions.loc[
            transactions['CustomerID'] == customer_id, 'ProductID'
        ]
    )
    return popular_products[
        ~popular_products['ProductID'].isin(purchased_products)
    ].head(number_of_recommendations)

display(popularity_recommendations(1))

,,,TotalRevenue,TotalPurchases
ProductID,Category,Brand,,
114,Fitness,TechMax,601047.68,1868
43,books,TechMax,582658.80,1340
69,Fitness,PrimeGoods,514355.20,5120
70,Fitness,PrimeGoods,472247.76,1132
102,home,PrimeGoods,441265.71,929
82,Kitchen,FitLife,365394.36,5493
47,Fitness,TechMax,354199.84,3044
55,Books,TechMax,345413.52,738
108,Fitness,HomePro,319176.06,703


,TotalRevenue,TotalPurchases
Category,,
Fitness,4158978.21,17231
Kitchen,1281807.05,9764
Gaming,1270882.36,4683
Books,1041279.66,3295
Electronics,917806.15,3501
Home,832020.85,4025
books,774439.58,2011
home,544234.56,1540
fitness,146495.46,630


,TotalRevenue,TotalPurchases
Brand,,
TechMax,3493986.09,12068
PrimeGoods,2466341.02,11006
FitLife,1908643.29,11876
NextGen,1326504.26,5699
HomePro,1034815.46,3196
SmartBuy,803594.02,3106


Category,Books,Electronics,Fitness,Gaming,Home,Kitchen,books,fitness,gaming,home,kitchen
Region,,,,,,,,,,,
East,247631.48,211653.26,945660.22,274033.03,207821.43,337158.19,125681.90,33951.64,7031.72,129142.37,9435.83
North,247753.76,228950.28,1031352.99,324394.28,217983.74,314158.69,247919.89,43057.22,5629.26,144649.92,8578.87
South,255267.77,219360.14,1052078.41,358950.93,214426.96,321216.50,158692.63,36998.48,11150.90,118944.75,8395.41
West,290626.65,257842.47,1129886.59,313504.12,191788.72,309273.67,242145.16,32488.12,9997.18,151497.52,5721.09


Category,Books,Electronics,Fitness,Gaming,Home,Kitchen,books,fitness,gaming,home,kitchen
LoyaltyLevel,,,,,,,,,,,
Bronze,521869.00,480017.64,2078274.80,588033.53,452944.56,671379.91,361348.17,71859.08,16469.68,292818.61,14500.66
Gold,231711.07,180802.81,833858.11,262943.62,150445.76,239516.44,136908.58,31552.83,7342.20,102609.02,5612.12
Silver,287699.59,256985.70,1246845.30,419905.21,228630.53,370910.70,276182.83,43083.55,9997.18,148806.93,12018.42


,ProductID,Category,Brand,TotalRevenue,TotalPurchases
5,82,Kitchen,FitLife,365394.36,5493
2,69,Fitness,PrimeGoods,514355.20,5120
6,47,Fitness,TechMax,354199.84,3044
0,114,Fitness,TechMax,601047.68,1868
1,43,books,TechMax,582658.80,1340


## Required evidence checklist
- [x] Data quality checks and integrated customer-product-transaction data
- [x] Customer-product interaction matrix
- [x] Top 10 customers by spend and purchase frequency
- [x] Top 10 products by revenue, category revenue and brand revenue
- [x] Three groupby analyses and two pivot tables
- [x] Popularity-based recommender
- [ ] Second recommendation approach
- [ ] Evaluation and examples for at least five customers